In [ ]:
""" Importing the required libraries and setting up the path to the PDF file. 
    calculating the total number of pages in the PDF file.
"""

import pymupdf
from pathlib import Path

PDF_PATH = Path("/Users/akhilteegala/Projects/Models_to_pipelines/RAG/Data/NASA_HANDBOOK.pdf")

pdf = pymupdf.open(PDF_PATH)

print("Total pages:", len(pdf))

Total pages: 25


In [ ]:
""" see whether PyMuPDF preserves meaningful paragraph boundaries, 
and whether the raw text contains unwanted whitespace, NULL characters, or repeated page headers."""

raw_blocks = []

for page_index in range(min(25, len(pdf))):
    page = pdf[page_index]

    for block in page.get_text("blocks", sort=True):
        x0, y0, x1, y1, text, block_no, block_type = block

        if block_type != 0 or not text.strip():
            continue

        raw_blocks.append({
            "page": page_index + 1,
            "bbox": (x0, y0, x1, y1),
            "text": text
        })

print("Total extracted blocks:", len(raw_blocks))

Total extracted blocks: 402


In [9]:
""" Check to see if the extracted blocks contain meaningful paragraph boundaries and 
whether they have unwanted whitespace, NULL characters, or repeated page headers. """

for block in raw_blocks:
    if block["page"] == 17:
        print(repr(block["text"][:200]))
        print("---")

'project’s systems engineering activities as performed by the technical team and directs, \ncommunicates, monitors, and coordinates tasks. The systems engineer reviews and evaluates the \ntechnical aspec'
---
'The systems engineer usually plays the key role in leading the development of the concept of \noperations (ConOps) and resulting system architecture, defining boundaries, defining and \nallocating requi'
---
'In summary, the systems engineer is skilled in the art and science of balancing organizational, \ncost, and technical interactions in complex systems. The systems engineer and supporting \norganization '
---
'Systems engineering plays a key role in the project organization. Managing a project consists of \nthree main objectives:  managing the technical aspects of the project, managing the project \nteam, and'
---
'This document focuses on the SE side of the diagram. The practices/processes are taken from \nNPR 7123.1, NASA Systems Engineering Processes and Requirements. Each

In [11]:
"""
Here we define a function to clean the text and removes 
unwanted whitespace, NULL characters, and repeated page headers.
"""
import re

def clean_text(text):
    """ 
    Cleans the text by removing unwanted whitespace, NULL characters.
    
    Args:
        text (str): The text to be cleaned. 
        
    Returns:
        str: The cleaned text.

    """
    # Remove NULL characters
    text = text.replace("\x00", "")

    # Normalize non-breaking spaces
    text = text.replace("\xa0", " ")

    # Replace line breaks and repeated whitespace with one space
    text = " ".join(text.split())

    return text.strip()


def is_repeated_footer(text):
    """ 
    Checks if the text is a repeated footer.    
    
    Args:
        text (str): The text to be checked.
        
        Returns:
        bool: True if the text is a repeated footer, False otherwise."""
    
    return bool(re.fullmatch(
        r"NASA Systems Engineering Handbook\s+Rev\s+2\s+[ivxlcdm\d]+",
        text,
        flags=re.IGNORECASE
    ))


cleaned_blocks = []

for block in raw_blocks:
    text = clean_text(block["text"])

    if not text or is_repeated_footer(text):
        continue

    cleaned_blocks.append({
        **block,
        "text": text
    })

print("Raw blocks:", len(raw_blocks))
print("Cleaned blocks:", len(cleaned_blocks))
print("Removed blocks:", len(raw_blocks) - len(cleaned_blocks))

Raw blocks: 402
Cleaned blocks: 377
Removed blocks: 25


In [20]:
page_blocks = [
    block for block in cleaned_blocks
    if block["page"] == 17
]

for i, block in enumerate(page_blocks):
    print(f"\nBlock {i + 1}")
    print("BBox:", tuple(round(x, 1) for x in block["bbox"]))
    print("Characters:", len(block["text"]))
    print("Text:", block["text"][:5000])


Block 1
BBox: (72.0, 72.3, 537.7, 140.8)
Characters: 422
Text: project’s systems engineering activities as performed by the technical team and directs, communicates, monitors, and coordinates tasks. The systems engineer reviews and evaluates the technical aspects of the project to ensure that the systems/subsystems engineering processes are functioning properly and evolves the system from concept to product. The entire technical team is involved in the systems engineering process.

Block 2
BBox: (72.0, 153.3, 530.8, 263.2)
Characters: 692
Text: The systems engineer usually plays the key role in leading the development of the concept of operations (ConOps) and resulting system architecture, defining boundaries, defining and allocating requirements, evaluating design tradeoffs, balancing technical risk between systems, defining and assessing interfaces, and providing oversight of verification and validation activities, as well as many other tasks. The systems engineer typically leads th

Block 1 begins with project’s systems engineering activities..., starting with a lowercase word. that means the starting of the line moght be in the page 16, So we inspect the page boundary.

If one paragraph spans two pages, treating each part as an independent paragraph could lose context.

We have two possible strategies:
- Preserve page boundaries: Simpler, but may split paragraphs.
- Merge across pages: Better paragraph continuity when the text truly continues, but requires preserving both source page numbers.

In [21]:
page_16 = [b for b in cleaned_blocks if b["page"] == 16]
page_17 = [b for b in cleaned_blocks if b["page"] == 17]

print("PAGE 16 - LAST BLOCK:")
print(page_16[-1]["text"][-400:])

print("\nPAGE 17 - FIRST BLOCK:")
print(page_17[0]["text"][:400])

PAGE 16 - LAST BLOCK:
1 Rechtin, Systems Architecting of Organizations: Why Eagles Can’t Swim. 2 Comments on systems engineering throughout Chapter 2.0 are extracted from the speech “System Engineering and the Two Cultures of Engineering” by Michael D. Griffin, NASA Administrator.

PAGE 17 - FIRST BLOCK:
project’s systems engineering activities as performed by the technical team and directs, communicates, monitors, and coordinates tasks. The systems engineer reviews and evaluates the technical aspects of the project to ensure that the systems/subsystems engineering processes are functioning properly and evolves the system from concept to product. The entire technical team is involved in the system


PyMuPDF extracted the page's text blocks, but the final block in reading order on page 16 was a footnote. The actual paragraph continuation may be in an earlier block.

So we need to inspect the body text separately from footnotes.

Key lesson
PDF extraction order isn't always the same as logical reading order.

In [22]:
page_16 = [
    b for b in cleaned_blocks
    if b["page"] == 16
]

for i, block in enumerate(page_16[-5:], start=len(page_16)-4):
    print(f"\nBlock {i}")
    print("BBox:", tuple(round(x, 1) for x in block["bbox"]))
    print("Text:", block["text"][-300:])


Block 2
BBox: (72.0, 116.9, 541.3, 295.8)
Text: erational performance requirements in the intended use environment over the planned life of the system within cost, schedule, and other constraints. It is a methodology that supports the containment of the life cycle cost of a system. In other words, systems engineering is a logical way of thinking.

Block 3
BBox: (72.0, 308.3, 536.3, 390.6)
Text: ein the contributions of structural engineers, electrical engineers, mechanism designers, power engineers, human factors engineers, and many more disciplines are evaluated and balanced, one against another, to produce a coherent whole that is not dominated by the perspective of a single discipline.2

Block 4
BBox: (72.0, 403.1, 538.9, 512.9)
Text:  of the operational system will be met. The art is in knowing when and where to probe. Personnel with these skills are usually tagged as “systems engineers.” They may have other titles—lead systems engineer, technical manager, chief engineer— but for 

Use PyMuPDF's layout information to identify paragraph candidates, but only merge blocks when there is sufficient evidence that the text continues.

In [23]:
page16_body = page_16[-2]
page17_first = page_17[0]

merged_text = (
    page16_body["text"].rstrip()
    + " "
    + page17_first["text"].lstrip()
)

print("Merged paragraph length:", len(merged_text))
print(merged_text[-500:])

Merged paragraph length: 1169
ems engineering approach is being followed. The systems engineer oversees the project’s systems engineering activities as performed by the technical team and directs, communicates, monitors, and coordinates tasks. The systems engineer reviews and evaluates the technical aspects of the project to ensure that the systems/subsystems engineering processes are functioning properly and evolves the system from concept to product. The entire technical team is involved in the systems engineering process.


In [24]:
for page_num in range(1, 25):
    current = [
        b for b in cleaned_blocks
        if b["page"] == page_num
        and b["bbox"][1] < 650
    ]

    next_page = [
        b for b in cleaned_blocks
        if b["page"] == page_num + 1
        and b["bbox"][1] < 650
    ]

    if not current or not next_page:
        continue

    last_text = current[-1]["text"]
    first_text = next_page[0]["text"]

    print(f"\nPages {page_num} → {page_num + 1}")
    print("Last:", repr(last_text[-100:]))
    print("First:", repr(first_text[:100]))


Pages 2 → 3
Last: 's of Tailoring and Customization ................................................................ 43'
First: '4.1.1 Process Description ..........................................................................'

Pages 3 → 4
Last: 'ng Guidance .................................................................................... 149'
First: '6.3 Interface Management ...........................................................................'

Pages 4 → 5
Last: '................................................................................................ 287'
First: 'Appendix Q: Reserved ...............................................................................'

Pages 5 → 6
Last: '................................................................................................ 324'
First: 'Part 1 Table of Figures'

Pages 6 → 7
Last: '.................................................................................................248'
First: 'Figure G.3-1 Techno

paragraph reconstruction is not simply about joining text across pages. We need to understand the type of content at each page boundary.

Layer 1 — Cleaned blocks

Preserve everything we extracted, including page number, bounding box, and cleaned text.

Layer 2 — Reconstructed paragraphs

Create paragraph records from body-text blocks, merging verified continuations and preserving their original page references.

This separation is important because we can always trace a reconstructed paragraph back to the original PDF.

In [25]:
from copy import deepcopy

paragraph_blocks = deepcopy(cleaned_blocks)

page16_body = next(
    b for b in paragraph_blocks
    if b["page"] == 16
    and b["text"].endswith("The systems engineer oversees the")
)

page17_first = next(
    b for b in paragraph_blocks
    if b["page"] == 17
    and b["text"].startswith("project’s systems engineering activities")
)

merged_record = {
    "pages": [16, 17],
    "text": page16_body["text"] + " " + page17_first["text"],
    "source_blocks": [
        {"page": 16, "bbox": page16_body["bbox"]},
        {"page": 17, "bbox": page17_first["bbox"]}
    ],
    "type": "paragraph"
}

print("Merged length:", len(merged_record["text"]))
print("Source pages:", merged_record["pages"])

Merged length: 1169
Source pages: [16, 17]


In [26]:
for p1, p2 in [(19, 20), (23, 24)]:
    print(f"\n{'=' * 15} Pages {p1} → {p2} {'=' * 15}")

    for page_num in [p1, p2]:
        blocks = [
            b for b in cleaned_blocks
            if b["page"] == page_num
        ]

        print(f"\nPage {page_num}")

        if page_num == p1:
            for b in blocks[-4:]:
                print("BBox:", tuple(round(v, 1) for v in b["bbox"]))
                print("END:", b["text"][-200:])
        else:
            for b in blocks[:4]:
                print("BBox:", tuple(round(v, 1) for v in b["bbox"]))
                print("START:", b["text"][:200])


=============== Pages 19 → 20 ===============

Page 19
BBox: (170.8, 549.1, 207.9, 559.5)
END: structure
BBox: (137.3, 605.4, 478.1, 618.8)
END: Figure 2.1-1 The Systems Engineering Engine (NPR 7123.1)
BBox: (72.0, 628.8, 508.2, 643.5)
END:  Product Realization Processes: The product realization processes are applied to each
BBox: (90.0, 643.9, 542.3, 698.6)
END: o create the design solution for each product (through buying, coding, building, or reusing) and to verify, validate, and transition up to the next hierarchical level those products that satisfy their

Page 20
BBox: (90.0, 72.3, 534.2, 99.4)
START: design solutions and meet stakeholder expectations as a function of the applicable life-cycle phase.
BBox: (72.0, 105.5, 505.2, 120.2)
START:  Technical Management Processes: The technical management processes are used to
BBox: (90.0, 120.6, 540.0, 175.3)
START: establish and evolve technical plans for the project, to manage communication across interfaces, to assess progress aga

Why are we using explicit merge rules?
Because our experiment has shown that:
- Footnotes can interrupt the extracted reading order.
- Figures can appear between related text blocks.
- A paragraph can continue onto another page.
- A list item can also continue across pages.
We don't yet have enough evidence to build a reliable general-purpose merging algorithm. Explicit rules let us establish a correct baseline.

In [27]:
from copy import deepcopy

paragraph_blocks = deepcopy(cleaned_blocks)

merge_rules = [
    (16, 17, "The systems engineer oversees the",
     "project’s systems engineering activities", "paragraph"),

    (19, 20, "those products that satisfy their",
     "design solutions and meet stakeholder expectations", "list_item"),

    (23, 24, "requirements are conveying; i.e., it",
     "may become more difficult to visualize", "paragraph")
]

merged_records = []

for p1, p2, ending, beginning, content_type in merge_rules:
    left = next(
        b for b in paragraph_blocks
        if b["page"] == p1 and b["text"].endswith(ending)
    )

    right = next(
        b for b in paragraph_blocks
        if b["page"] == p2 and b["text"].startswith(beginning)
    )

    merged_records.append({
        "pages": [p1, p2],
        "text": left["text"] + " " + right["text"],
        "source_blocks": [
            {"page": p1, "bbox": left["bbox"]},
            {"page": p2, "bbox": right["bbox"]}
        ],
        "type": content_type
    })

for record in merged_records:
    print(record["pages"], record["type"], len(record["text"]))

[16, 17] paragraph 1169
[19, 20] list_item 470
[23, 24] paragraph 1264


In [28]:
merge_lookup = {}
skip_ids = set()

for record in merged_records:
    source_keys = [
        (source["page"], tuple(source["bbox"]))
        for source in record["source_blocks"]
    ]

    merge_lookup[source_keys[0]] = record
    skip_ids.update(source_keys)

structured_dataset = []

for block in cleaned_blocks:
    key = (block["page"], tuple(block["bbox"]))

    if key in merge_lookup:
        structured_dataset.append(merge_lookup[key])

    elif key in skip_ids:
        continue

    else:
        structured_dataset.append({
            "pages": [block["page"]],
            "text": block["text"],
            "source_blocks": [
                {
                    "page": block["page"],
                    "bbox": block["bbox"]
                }
            ],
            "type": block.get("type", "unclassified")
        })

print("Original cleaned blocks:", len(cleaned_blocks))
print("Final dataset records:", len(structured_dataset))
print("Merged records:", len(merged_records))

Original cleaned blocks: 377
Final dataset records: 374
Merged records: 3


In [29]:
original_chars = sum(
    len(block["text"])
    for block in cleaned_blocks
)

final_chars = sum(
    len(record["text"])
    for record in structured_dataset
)

print("Original characters:", original_chars)
print("Final characters:", final_chars)
print("Difference:", final_chars - original_chars)

assert len(structured_dataset) == len(cleaned_blocks) - 3
assert final_chars - original_chars == 3

print("Validation passed!")

Original characters: 60912
Final characters: 60915
Difference: 3
Validation passed!


In [31]:
from collections import Counter

for i, record in enumerate(structured_dataset, start=1):
    record["record_id"] = f"nasa_{i:04d}"

print("Total records:", len(structured_dataset))

print("\nContent types:")
print(Counter(record["type"] for record in structured_dataset))

print("\nSample record:")
print(structured_dataset[2])

Total records: 374

Content types:
Counter({'unclassified': 371, 'paragraph': 2, 'list_item': 1})

Sample record:
{'pages': [2], 'text': 'Acknowledgments.......................................................................................................................... xi', 'source_blocks': [{'page': 2, 'bbox': (72.0, 110.98799133300781, 542.0015258789062, 124.42499542236328)}], 'type': 'unclassified', 'record_id': 'nasa_0003'}


In [32]:
for record in structured_dataset:
    if record["type"] == "unclassified" and 16 in record["pages"]:
        print(
            record["record_id"],
            "|",
            repr(record["text"][:120])
        )

nasa_0225 | '2.0 Fundamentals of Systems Engineering'
nasa_0226 | 'At NASA, “systems engineering” is defined as a methodical, multi-disciplinary approach for the design, realization, tech'
nasa_0227 | 'Systems engineering is the art and science of developing an operable system capable of meeting requirements within often'
nasa_0228 | 'Systems engineering seeks a safe and balanced design in the face of opposing interests and multiple, sometimes conflicti'
nasa_0230 | '1 Rechtin, Systems Architecting of Organizations: Why Eagles Can’t Swim. 2 Comments on systems engineering throughout Ch'


In [33]:
import re

def classify_record(record):
    text = record["text"].strip()

    # Preserve verified classifications
    if record["type"] != "unclassified":
        return record["type"]

    # Section headings such as 2.0 or 2.1.1
    if re.match(r"^\d+(?:\.\d+)+\s+[A-Z]", text) and len(text) < 120:
        return "heading_candidate"

    # Figure captions
    if re.match(r"^Figure\s+\d+(?:\.\d+)*-\d+", text, re.I):
        return "figure_reference_candidate"

    # Bulleted content
    if re.match(r"^[•\uf0b7]", text):
        return "list_candidate"

    # Long prose-like blocks
    if len(text) > 150:
        return "paragraph_candidate"

    return "other"

for record in structured_dataset:
    record["type"] = classify_record(record)

In [34]:
from collections import Counter

print(Counter(r["type"] for r in structured_dataset))

for record in structured_dataset:
    if 16 in record["pages"]:
        print(record["record_id"], record["type"], repr(record["text"][:90]))

Counter({'other': 276, 'paragraph_candidate': 51, 'heading_candidate': 30, 'list_candidate': 9, 'figure_reference_candidate': 5, 'paragraph': 2, 'list_item': 1})
nasa_0225 heading_candidate '2.0 Fundamentals of Systems Engineering'
nasa_0226 paragraph_candidate 'At NASA, “systems engineering” is defined as a methodical, multi-disciplinary approach for'
nasa_0227 paragraph_candidate 'Systems engineering is the art and science of developing an operable system capable of mee'
nasa_0228 paragraph_candidate 'Systems engineering seeks a safe and balanced design in the face of opposing interests and'
nasa_0229 paragraph 'The exact role and responsibility of the systems engineer may change from project to proje'
nasa_0230 paragraph_candidate '1 Rechtin, Systems Architecting of Organizations: Why Eagles Can’t Swim. 2 Comments on sys'


In [35]:
page = pdf[15]  # PDF page 16

page_dict = page.get_text("dict")

for block in page_dict["blocks"]:
    if block["type"] != 0:
        continue

    for line in block["lines"]:
        for span in line["spans"]:
            text = span["text"].strip()

            if not text:
                continue

            print(
                f"Font size: {span['size']:.1f}",
                f"Font: {span['font']}",
                f"Y: {span['bbox'][1]:.1f}",
                f"Text: {text[:80]}"
            )

Font size: 12.0 Font: TimesNewRomanPSMT Y: 716.7 Text: NASA Systems Engineering Handbook    Rev 2
Font size: 12.0 Font: TimesNewRomanPSMT Y: 716.7 Text:  
Font size: 12.0 Font: TimesNewRomanPSMT Y: 716.7 Text: 3
Font size: 14.0 Font: Arial-BoldMT Y: 84.4 Text: 2.0 Fundamentals of Systems Engineering
Font size: 12.0 Font: TimesNewRomanPSMT Y: 116.9 Text: At NASA, “systems engineering” is defined as a methodical, multi-disciplinary ap
Font size: 12.0 Font: TimesNewRomanPSMT Y: 130.7 Text: design, realization, technical management, operations, and retirement of a syste
Font size: 12.0 Font: TimesNewRomanPSMT Y: 144.5 Text: the combination of elements that function together to produce the capability req
Font size: 12.0 Font: TimesNewRomanPSMT Y: 158.3 Text: need. The elements include all hardware, software, equipment, facilities, person
Font size: 12.0 Font: TimesNewRomanPSMT Y: 172.1 Text: and procedures needed for this purpose; that is, all things required to produce 
Font size: 12.0 Fon

PyMuPDF is giving us something regex cannot: typographical information about the text.

PyMuPDF can extract text at a more detailed level:

Page → Blocks → Lines → Spans

A span is a run of text with consistent font properties, including font size and font name.

Notice that the footer and body both use 12 pt text. This proves why font size alone is insufficient. We need to combine typography, location, and text patterns.

In [36]:
from collections import Counter

def get_block_font_info(page, target_bbox):
    page_dict = page.get_text("dict")

    font_sizes = []
    font_names = []

    for block in page_dict["blocks"]:
        if block["type"] != 0:
            continue

        if not pymupdf.Rect(block["bbox"]).intersects(
            pymupdf.Rect(target_bbox)
        ):
            continue

        for line in block["lines"]:
            for span in line["spans"]:
                if not span["text"].strip():
                    continue

                font_sizes.append(round(span["size"], 1))
                font_names.append(span["font"])

    return {
        "font_size": Counter(font_sizes).most_common(1)[0][0]
        if font_sizes else None,
        "font": Counter(font_names).most_common(1)[0][0]
        if font_names else None
    }

In [37]:
for record in structured_dataset:
    if 16 not in record["pages"]:
        continue

    source = next(
        s for s in record["source_blocks"]
        if s["page"] == 16
    )

    info = get_block_font_info(
        pdf[15],
        source["bbox"]
    )

    print(
        record["record_id"],
        record["type"],
        info,
        repr(record["text"][:65])
    )

nasa_0225 heading_candidate {'font_size': 14.0, 'font': 'Arial-BoldMT'} '2.0 Fundamentals of Systems Engineering'
nasa_0226 paragraph_candidate {'font_size': 12.0, 'font': 'TimesNewRomanPSMT'} 'At NASA, “systems engineering” is defined as a methodical, multi-'
nasa_0227 paragraph_candidate {'font_size': 12.0, 'font': 'TimesNewRomanPSMT'} 'Systems engineering is the art and science of developing an opera'
nasa_0228 paragraph_candidate {'font_size': 12.0, 'font': 'TimesNewRomanPSMT'} 'Systems engineering seeks a safe and balanced design in the face '
nasa_0229 paragraph {'font_size': 12.0, 'font': 'TimesNewRomanPSMT'} 'The exact role and responsibility of the systems engineer may cha'
nasa_0230 paragraph_candidate {'font_size': 10.0, 'font': 'TimesNewRomanPSMT'} '1 Rechtin, Systems Architecting of Organizations: Why Eagles Can’'


Now we are building the classifier - layout aware 
intially when we went and tried to classify only using the paragraph or scentence length we got the info but some of the foot notes are also comming as paragraphs because they are more than the character length we specified 

then we went to text blocks span there w are abel to see and figure out the font size and we are abel to identigy the paragraph canidates used a constant 12 and the headings as 14 and also rhe foot as 10 but it still say paragraph 

In [ ]:
"""
this is a layout aware classifier that uses font size, font name, and bounding box location to classify the text blocks into different categories such as 
footnotes, headings, figure captions, lists, and paragraphs. It preserves manually verified merged records and applies specific rules based on typographical 
information to determine the type of content in the PDF.

"""
import re

def classify_with_layout(record, pdf):

    # Preserve manually verified merged records
    if record["type"] in ("paragraph", "list_item"):
        return record["type"]

    text = record["text"].strip()

    source = record["source_blocks"][0]
    page = pdf[source["page"] - 1]

    bbox = source["bbox"]

    font_info = get_block_font_info(page, bbox)

    font_size = font_info["font_size"]
    font_name = font_info["font"] or ""

    if font_size is None:
        return "unknown"

    # Rule 1: Footnotes
    if (
        font_size <= 10
        and bbox[1] > page.rect.height * 0.75
    ):
        return "footnote_candidate"

    # Rule 2: Section headings
    if (
        font_size >= 14
        and "bold" in font_name.lower()
        and len(text) < 150
    ):
        return "heading_candidate"

    # Rule 3: Figure captions
    if re.match(r"^Figure\s+\d+(?:\.\d+)*-\d+", text, re.I):
        return "figure_reference_candidate"

    # Rule 4: Lists
    if re.match(r"^[•\uf0b7]", text):
        return "list_candidate"

    # Rule 5: Body paragraphs
    if (
        abs(font_size - 12) < 0.1
        and len(text) > 150
    ):
        return "paragraph_candidate"

    return "other"

In [ ]:
"""instead of over writing the previous classification label now we are going to add a 
new key called layout_type and we are going to classify the records based on the layout of the text in the pdf file."""
for record in structured_dataset:
    record["layout_type"] = classify_with_layout(record, pdf)

In [40]:
from collections import Counter

print("Version 1 - Text only:")
print(Counter(r["type"] for r in structured_dataset))

print("\nVersion 2 - Text + Layout:")
print(Counter(r["layout_type"] for r in structured_dataset))

Version 1 - Text only:
Counter({'other': 276, 'paragraph_candidate': 51, 'heading_candidate': 30, 'list_candidate': 9, 'figure_reference_candidate': 5, 'paragraph': 2, 'list_item': 1})

Version 2 - Text + Layout:
Counter({'other': 290, 'paragraph_candidate': 44, 'heading_candidate': 13, 'footnote_candidate': 10, 'list_candidate': 9, 'figure_reference_candidate': 5, 'paragraph': 2, 'list_item': 1})


In [41]:
changed = [
    r for r in structured_dataset
    if r["type"] != r["layout_type"]
]

print("Total changed:", len(changed))

for record in changed:
    print(
        record["record_id"],
        "Pages:", record["pages"],
        "|", record["type"],
        "→", record["layout_type"],
        "|", repr(record["text"][:100])
    )

Total changed: 42
nasa_0008 Pages: [2] | heading_candidate → other | '2.1 The Common Technical Processes and the SE Engine ...............................................'
nasa_0009 Pages: [2] | heading_candidate → other | '2.2 An Overview of the SE Engine by Project Phase ..................................................'
nasa_0011 Pages: [2] | heading_candidate → other | '2.4 Distinctions between Product Verification and Product Validation ...............................'
nasa_0013 Pages: [2] | heading_candidate → other | '2.6 Human Systems Integration (HSI) in the SE Process ..............................................'
nasa_0014 Pages: [2] | heading_candidate → other | '2.7 Competency Model for Systems Engineers .........................................................'
nasa_0019 Pages: [2] | heading_candidate → other | '3.4 Project Phase A: Concept and Technology Development ............................................'
nasa_0020 Pages: [2] | heading_candidate → other | '3.5 Pr

In [42]:
def is_toc_entry(text):
    return bool(
        re.match(
            r"^\d+(?:\.\d+)+\s+.+\.{5,}\s*\d*\s*$",
            text
        )
    )

In [43]:
for record in changed:
    if record["type"] == "heading_candidate" and record["layout_type"] == "other":
        print(
            record["record_id"],
            is_toc_entry(record["text"]),
            repr(record["text"][:85])
        )

nasa_0008 True '2.1 The Common Technical Processes and the SE Engine ................................'
nasa_0009 True '2.2 An Overview of the SE Engine by Project Phase ...................................'
nasa_0011 True '2.4 Distinctions between Product Verification and Product Validation ................'
nasa_0013 True '2.6 Human Systems Integration (HSI) in the SE Process ...............................'
nasa_0014 True '2.7 Competency Model for Systems Engineers ..........................................'
nasa_0019 True '3.4 Project Phase A: Concept and Technology Development .............................'
nasa_0020 True '3.5 Project Phase B: Preliminary Design and Technology Completion ...................'
nasa_0022 True '3.7 Project Phase D: System Assembly, Integration and Test, Launch ..................'
nasa_0023 True '3.8 Project Phase E: Operations and Sustainment .....................................'
nasa_0026 True '3.11 Tailoring and Customization of NPR 7123.1 Requiremen

In [44]:
def classify_v3(record, pdf):

    text = record["text"].strip()

    # Preserve manually verified cross-page merges
    if record["type"] in ("paragraph", "list_item"):
        return record["type"]

    # Identify TOC entries before heading classification
    if is_toc_entry(text):
        return "toc_entry"

    return classify_with_layout(record, pdf)

In [45]:
for record in structured_dataset:
    record["v3_type"] = classify_v3(record, pdf)

print(Counter(r["v3_type"] for r in structured_dataset))

Counter({'other': 207, 'toc_entry': 83, 'paragraph_candidate': 44, 'heading_candidate': 13, 'footnote_candidate': 10, 'list_candidate': 9, 'figure_reference_candidate': 5, 'paragraph': 2, 'list_item': 1})


In [46]:
for record in structured_dataset:
    if (
        record["layout_type"] == "footnote_candidate"
        and record["record_id"] != "nasa_0230"
    ):
        source = record["source_blocks"][0]
        page = pdf[source["page"] - 1]

        info = get_block_font_info(page, source["bbox"])

        print(
            record["record_id"],
            "page:", source["page"],
            "font:", info["font_size"],
            "y:", round(source["bbox"][1], 1),
            "page_height:", round(page.rect.height, 1),
            "text:", repr(record["text"][:90])
        )

nasa_0315 page: 20 font: 10.0 y: 606.4 page_height: 792.0 text: 'Product Verification Verification'
nasa_0316 page: 20 font: 10.0 y: 622.3 page_height: 792.0 text: 'Product Validation Validation'
nasa_0317 page: 20 font: 10.0 y: 638.3 page_height: 792.0 text: 'Product Transition Control of Work Transfers; Post Delivery Support, Preservation of Produ'
nasa_0318 page: 20 font: 10.0 y: 665.8 page_height: 792.0 text: 'Technical Planning Planning of Product Realization; Review of Requirements; Measurement, A'
nasa_0319 page: 20 font: 10.0 y: 693.4 page_height: 792.0 text: 'Requirements Management Design and Development Planning; Purchasing'
nasa_0346 page: 22 font: 10.0 y: 612.0 page_height: 792.0 text: 'Implementation'
nasa_0347 page: 22 font: 10.0 y: 651.3 page_height: 792.0 text: 'Operations-ready system end product with supporting related enabling products'
nasa_0348 page: 22 font: 10.0 y: 651.3 page_height: 792.0 text: 'Phase D System Assembly, Integration and Test, Launch'
nasa_0349 p

In [47]:
for page_num in [16, 20, 22]:
    page = pdf[page_num - 1]

    tables = page.find_tables()

    print(f"\nPage {page_num}")
    print("Tables detected:", len(tables.tables))

    for i, table in enumerate(tables.tables, start=1):
        print(
            f"Table {i}",
            "bbox:", tuple(round(v, 1) for v in table.bbox),
            "rows:", table.row_count,
            "columns:", table.col_count
        )


Page 16
Tables detected: 0

Page 20
Tables detected: 1
Table 1 bbox: (72.2, 492.0, 539.8, 706.9) rows: 12 columns: 6

Page 22
Tables detected: 1
Table 1 bbox: (77.8, 359.1, 581.7, 711.0) rows: 6 columns: 4


In [48]:
def is_inside_table(page, bbox, threshold=0.5):
    block_rect = pymupdf.Rect(bbox)

    if block_rect.is_empty:
        return False

    tables = page.find_tables()

    for table in tables.tables:
        table_rect = pymupdf.Rect(table.bbox)
        intersection = block_rect & table_rect

        if intersection.is_empty:
            continue

        overlap = intersection.get_area() / block_rect.get_area()

        if overlap >= threshold:
            return True

    return False

In [49]:
def classify_v4(record, pdf):

    # Preserve verified cross-page merges
    if record["type"] in ("paragraph", "list_item"):
        return record["type"]

    text = record["text"].strip()

    source = record["source_blocks"][0]
    page = pdf[source["page"] - 1]
    bbox = source["bbox"]

    # Detect table content before footnotes
    if is_inside_table(page, bbox):
        return "table_content"

    # Detect TOC entries
    if is_toc_entry(text):
        return "toc_entry"

    # Reuse layout-aware classification
    return classify_with_layout(record, pdf)

In [50]:
for record in structured_dataset:
    record["v4_type"] = classify_v4(record, pdf)

print(Counter(r["v4_type"] for r in structured_dataset))

Counter({'other': 172, 'toc_entry': 83, 'paragraph_candidate': 44, 'table_content': 44, 'heading_candidate': 13, 'list_candidate': 9, 'figure_reference_candidate': 5, 'paragraph': 2, 'footnote_candidate': 1, 'list_item': 1})


In [51]:
check_ids = [
    "nasa_0230",  # Genuine footnote
    "nasa_0315",  # Table content
    "nasa_0319",  # Table content
    "nasa_0346",  # Table content
    "nasa_0349"   # Table content
]

for record in structured_dataset:
    if record["record_id"] in check_ids:
        print(
            record["record_id"],
            "| V2:", record["layout_type"],
            "| V4:", record["v4_type"]
        )

nasa_0230 | V2: footnote_candidate | V4: footnote_candidate
nasa_0315 | V2: footnote_candidate | V4: table_content
nasa_0319 | V2: footnote_candidate | V4: table_content
nasa_0346 | V2: footnote_candidate | V4: table_content
nasa_0349 | V2: footnote_candidate | V4: table_content


In [52]:
from collections import defaultdict

others_by_page = defaultdict(list)

for record in structured_dataset:
    if record["v4_type"] == "other":
        others_by_page[record["pages"][0]].append(record)

for page_num in sorted(others_by_page):
    records = others_by_page[page_num]

    print(f"\nPage {page_num}: {len(records)} other records")

    for record in records[:3]:
        print(
            record["record_id"],
            repr(record["text"][:100])
        )
        


Page 2: 2 other records
nasa_0001 'Part 1 Table of Contents'
nasa_0003 'Acknowledgments.....................................................................................'

Page 4: 16 other records
nasa_0087 'Appendix A: Acronyms ...............................................................................'
nasa_0088 'Appendix B: Glossary ...............................................................................'
nasa_0089 'Appendix C: How to Write a Good Requirement - Checklist ............................................'

Page 5: 6 other records
nasa_0103 'Appendix Q: Reserved ...............................................................................'
nasa_0104 'Appendix R: HSI Plan Content Outline ...............................................................'
nasa_0105 'Appendix S: Concept of Operations Annotated Outline.................................................'

Page 6: 1 other records
nasa_0109 'Part 1 Table of Figures'

Page 8: 1 other records
nasa_0115 

In [53]:
for record in structured_dataset:
    if record["record_id"] in ("nasa_0326", "nasa_0369"):
        source = record["source_blocks"][0]
        page = pdf[source["page"] - 1]

        print("\nRecord:", record["record_id"])
        print("Page:", source["page"])
        print("Text:", record["text"])
        print("BBox:", source["bbox"])
        print("Font:", get_block_font_info(page, source["bbox"]))
        print("V4 classification:", record["v4_type"])


Record: nasa_0326
Page: 21
Text: Decision Analysis Measurement, Analysis and Improvement; Analysis of Data
BBox: (72.0, 193.7318878173828, 520.5651245117188, 223.5237579345703)
Font: {'font_size': 10.0, 'font': 'ArialMT'}
V4 classification: other

Record: nasa_0369
Page: 24
Text: meet each “shall” statement as proven though performance of a test, analysis, inspection, or demonstration (or combination of these).
BBox: (90.0, 682.0679931640625, 533.0157470703125, 709.1519775390625)
Font: {'font_size': 12.0, 'font': 'TimesNewRomanPSMT'}
V4 classification: other


In [54]:
page = pdf[20]

tables = page.find_tables()

print("Tables:", len(tables.tables))

for table in tables.tables:
    print("BBox:", table.bbox)

Tables: 1
BBox: (72.27000522613525, 72.43203125, 539.7300109863281, 207.239990234375)


In [55]:
index = next(
    i for i, r in enumerate(structured_dataset)
    if r["record_id"] == "nasa_0369"
)

for record in structured_dataset[max(0, index-3):index+2]:
    print(
        record["record_id"],
        record["pages"],
        record["v4_type"],
        repr(record["text"][:180])
    )

nasa_0366 [24] heading_candidate '2.4 Distinctions between Product Verification and Product Validation'
nasa_0367 [24] paragraph_candidate 'From a process perspective, the Product Verification and Product Validation Processes may be similar in nature, but the objectives are fundamentally different:'
nasa_0368 [24] list_candidate '\uf0b7 Verification of a product shows proof of compliance with requirements—that the product can'
nasa_0369 [24] other 'meet each “shall” statement as proven though performance of a test, analysis, inspection, or demonstration (or combination of these).'
nasa_0370 [25] list_candidate '\uf0b7 Validation of a product shows that the product accomplishes the intended purpose in the'


In [56]:
from copy import deepcopy

rag_dataset = deepcopy(structured_dataset)

left = next(r for r in rag_dataset if r["record_id"] == "nasa_0368")
right = next(r for r in rag_dataset if r["record_id"] == "nasa_0369")

merged = {
    **left,
    "text": left["text"].rstrip() + " " + right["text"].lstrip(),
    "source_blocks": left["source_blocks"] + right["source_blocks"],
    "pages": sorted(set(left["pages"] + right["pages"])),
    "v4_type": "list_item"
}

rag_dataset = [
    merged if r["record_id"] == "nasa_0368" else r
    for r in rag_dataset
    if r["record_id"] != "nasa_0369"
]

print("Records after merge:", len(rag_dataset))
print("Merged text:", merged["text"])

Records after merge: 373
Merged text:  Verification of a product shows proof of compliance with requirements—that the product can meet each “shall” statement as proven though performance of a test, analysis, inspection, or demonstration (or combination of these).


In [57]:
INCLUDE_TYPES = {
    "heading_candidate",
    "paragraph_candidate",
    "paragraph",
    "list_candidate",
    "list_item"
}

In [58]:
retrieval_dataset = [
    {
        "record_id": record["record_id"],
        "pages": record["pages"],
        "text": record["text"],
        "content_type": record["v4_type"],
        "source_blocks": record["source_blocks"]
    }
    for record in rag_dataset
    if record["v4_type"] in INCLUDE_TYPES
]

In [59]:
print("Full dataset:", len(rag_dataset))
print("Retrieval dataset:", len(retrieval_dataset))

print(
    Counter(r["content_type"] for r in retrieval_dataset)
)

Full dataset: 373
Retrieval dataset: 69
Counter({'paragraph_candidate': 44, 'heading_candidate': 13, 'list_candidate': 8, 'paragraph': 2, 'list_item': 2})


In [68]:
%pip install pandas 
import pandas as pd

df = pd.DataFrame(retrieval_dataset)

display(df[["record_id", "pages", "content_type", "text"]])

Note: you may need to restart the kernel to use updated packages.


,record_id,pages,content_type,text
0,nasa_0002,[2],paragraph_candidate,Preface .........................................
1,nasa_0111,[6],paragraph_candidate,Flow for Flight and Ground Systems Accompanyin...
2,nasa_0112,[6],paragraph_candidate,Aspects of Effectiveness ........................
3,nasa_0113,[6],paragraph_candidate,Continuous Risk Management (Source: NASA/SP-20...
4,nasa_0114,[7],paragraph_candidate,Figure G.3-1 Technology Assessment Process ......
...,...,...,...,...
64,nasa_0370,[25],list_candidate, Validation of a product shows that the produ...
65,nasa_0371,[25],paragraph_candidate,intended environment—that it meets the expecta...
66,nasa_0372,[25],paragraph_candidate,Verification testing relates back to the appro...
67,nasa_0373,[25],paragraph_candidate,Validation relates back to the ConOps document...


In [69]:
record = retrieval_dataset[0]

print("Record ID:", record["record_id"])
print("Pages:", record["pages"])
print("Type:", record["content_type"])
print("Text:\n", record["text"])

Record ID: nasa_0002
Pages: [2]
Type: paragraph_candidate
Text:
 Preface ............................................................................................................................................. x


In [70]:
record = next(
    r for r in retrieval_dataset
    if r["record_id"] == "nasa_0229"
)

print(record["text"])

The exact role and responsibility of the systems engineer may change from project to project depending on the size and complexity of the project and from phase to phase of the life cycle. For large projects, there may be one or more systems engineers. For small projects, the project manager may sometimes perform these practices. But whoever assumes those responsibilities, the systems engineering functions should be performed. The actual assignment of the roles and responsibilities of the named systems engineer may also therefore vary. The lead systems engineer ensures that the system technically fulfills the defined needs and requirements and that a proper systems engineering approach is being followed. The systems engineer oversees the project’s systems engineering activities as performed by the technical team and directs, communicates, monitors, and coordinates tasks. The systems engineer reviews and evaluates the technical aspects of the project to ensure that the systems/subsystems